<a href="https://colab.research.google.com/github/DanielLucero25/CSC349/blob/main/HW6/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 349 HW6: Analysis


## 1. Components and reconstruction quality

The PCA fitted to all 400 images needed 123 components to keep 95 percent of the total variance. That is about 3 percent of the original 4096 dimensions. It removes about 97 percent of the dimensions while keeping most of the variation in these images. This is possible because nearby pixels and different parts of the face often change together. The model does not need a separate independent feature for every pixel. The PCA fitted only on the 320 training images needed 111 components. These counts are different because the two PCA models learned from different sets of images.

At k = 10 the faces are blurry. The general shape and large light and dark areas remain but the eyes and mouth lose detail. The glasses on subject 16 are much harder to see. The facial hair on subjects 16 and 24 also loses its shape. At k = 50 the faces are easier to recognize and the glasses and facial hair become clearer. At k = 100 more detail returns but there are still differences around the eyes and face edges.

At k = 200 the reconstructions look very close to the originals at the size shown. At k = 300 they are difficult to tell apart by looking at the grid. I would use around 300 as the point where these examples appear almost indistinguishable. They are still not exact copies. The average MSE for the five faces falls from about 0.005764 at k = 10 to 0.000127 at k = 300. Even at k = 300 there is a small error. These are images that PCA saw during fitting so this does not measure reconstruction quality on completely new faces.

## 2. What the eigenfaces capture

The first eigenfaces show broad patterns across the face. Large bright and dark areas show changes in lighting and shadows. Other patterns show differences around the eyes, nose, cheeks and mouth. They can also reflect head position and face shape. The first component explains about 24 percent of the total variance so one large pattern accounts for a lot of the differences between these images. An eigenface is not one actual person. It is a pattern that can be added to or subtracted from the average face.

PCA puts the patterns with the most variance first. After the broad patterns have been included the later components describe smaller changes. These can include skin texture, hair, glasses edges and small changes in shadows. Each later component explains less of the total variation. The patterns can look more scattered and less like a whole face.

Later components can also include camera noise and small changes that do not help identify a person. However a low variance component is not automatically noise. It might contain a small detail that separates two subjects. PCA uses pixel variation and does not use subject labels to decide which details matter. Removing components can make a picture smoother but can also remove useful information.

## 3. Why PCA must use only the training split

PCA learns the average face and the directions of variation from the data it is fitted on. If I fit PCA to the full dataset before splitting it then the test images help decide how all images are represented. This lets information from the test set enter the training process. It is called data leakage even though PCA does not use the subject labels.

The test set is supposed to act like new data that was not available during fitting. Letting its images affect the PCA directions or the number of retained components makes that check less independent. The accuracy can look better than it would on truly new images. Leakage does not guarantee that the score will rise on every split but it makes the evaluation less trustworthy.

In Tasks 5 through 7 I split the data first. Each classification pipeline fits its PCA step using only X_train. It then transforms X_test using the already fitted PCA without learning anything new from X_test. The full-data PCA from Task 2 is used only for exploring variance and showing images. I do not pass that fitted model into the classification pipelines. If I later tune model settings I should also fit PCA separately within each training fold during cross-validation.

## 4. Comparing the PCA classifiers

Logistic regression on PCA features had 96.25 percent accuracy. The RBF SVC had 93.75 percent and the soft voting ensemble had 95 percent. There were 80 test images. These scores mean logistic regression got 77 correct, the SVC got 75 correct and soft voting got 76 correct. The non-linear kernel and the ensemble did not give a substantial gain in this run. They actually scored slightly lower than logistic regression. The differences are only one or two images so I would not say one method is always better based on this split.

The RBF kernel can separate curved patterns but that extra flexibility was not enough to improve the result here. The ensemble averages probabilities from two SVMs and a random forest. That can help when the models make different useful predictions. It can also reduce accuracy when weaker predictions pull the average away from the right subject. These were simple model settings rather than settings chosen through a larger search. The results do not show that tuning could never help.

All these classifiers work with individual pixel values or global PCA features. They do not directly learn that a group of nearby pixels forms an eye or a mouth. A small shift in a face can change many pixel positions even when it is the same person. Lighting and expression can also change the values without changing identity. PCA keeps directions with large variance but those directions may describe lighting more than identity. Keeping 95 percent of variance does not guarantee keeping 95 percent of the useful information for classification.

The scores are high because this is a small collection of similarly sized and aligned faces. There are only eight training images and two test images per person. This experiment cannot establish an exact performance ceiling. It does show why strong results on this dataset should not be treated as proof that the models can handle large changes in pose, backgrounds or lighting.

## 5. Accuracy and training efficiency

Raw logistic regression had the highest accuracy at 97.5 percent and took about 3.830 seconds to fit. PCA logistic regression had 96.25 percent accuracy and took about 0.287 seconds. The RBF SVC took about 0.230 seconds with 93.75 percent accuracy. Soft voting took about 0.662 seconds with 95 percent accuracy. These times include fitting PCA for the PCA pipelines. Prediction time is not included. They are single measurements and could change when the notebook is run in Colab.

PCA logistic regression was a useful tradeoff in this run. It reduced the input to 111 components and fit much faster than raw logistic regression while getting only one fewer test image correct. The ensemble needed more training time than the other PCA models without getting the best accuracy. Part of that extra work comes from fitting three classifiers and estimating probabilities for the SVMs. More models did not automatically make the predictions better.

Even though PCA helped with speed it is still a global linear transformation. It learns one set of directions for the whole image. It does not build up useful local features or make them reliable when a face moves to a different position. The classifiers then have to work with that limited representation. As images get larger and more varied this becomes a harder problem. Important details may be dropped while large changes in brightness are kept.

For more complex image tasks a convolutional neural network can learn local patterns such as edges and combine them into parts of a face. Training on more varied images and using image augmentation can also help with changes in position and lighting. That does not mean a neural network will always perform better with only 400 images. A pretrained vision model would be another option to test. My results support PCA as a simple compression tool for these aligned faces but they do not show that it solves general computer vision problems.

## References

[Scikit-Learn Olivetti Faces dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_olivetti_faces.html)

[Scikit-Learn PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)

[Course dimensionality reduction example](https://github.com/bforoura/ML26/blob/main/Module7_Dimensionality/plant_classification.ipynb)